# 20-тарау. Генеративті ЖИ: қауіпсіздік, құқық және жауапкершілік

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 20.1. Жаңа мүмкіндіктер, жаңа сұрақтар

## 20.2. Дипфейктер және алаяқтық

## 20.3. ЖИ жазған мәтінді қалай танимыз?

In [ ]:
import math
import random
import urllib.request
import torch

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
for path in ["data/kitap.txt", "vol2/gpt.py", "vol2/gpt_kitap.pt"]:
    urllib.request.urlretrieve(RAW + path, path.split("/")[-1])
from gpt import GPT, GPTConfig

text = open("kitap.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)
V = len(chars)

model = GPT(GPTConfig(vocab_size=V, block_size=64))
model.load_state_dict(torch.load("gpt_kitap.pt"))
model.eval()
print("сөздік:", V, "таңба")

In [ ]:
GAMMA, KEY = 0.5, 2026
GREEN = torch.zeros(V, V, dtype=torch.bool)
for prev in range(V):
    g = torch.Generator().manual_seed(KEY * 1000 + prev)
    GREEN[prev, torch.randperm(V, generator=g)[:int(GAMMA * V)]] = True

@torch.no_grad()
def generate(prompt, n=300, delta=0.0, seed=0):
    torch.manual_seed(seed)
    ids = torch.tensor([encode(prompt)])
    for _ in range(n):
        logits = model(ids[:, -64:])[0][0, -1]
        logits = logits + delta * GREEN[ids[0, -1]]     # су таңба
        nxt = torch.multinomial(torch.softmax(logits, -1), 1)
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return decode(ids[0, len(prompt):].tolist())

In [ ]:
def z_score(s):
    ids = encode(s)
    green = sum(GREEN[a, b].item() for a, b in zip(ids, ids[1:]))
    T = len(ids) - 1
    return ((green - GAMMA * T) / math.sqrt(T * GAMMA * (1 - GAMMA)),
            green / T)

prompt = "Нейрондық желі"
normal_text = generate(prompt, delta=0.0, seed=1)
wm_text = generate(prompt, delta=2.0, seed=1)
human_text = text[50000:50300]
for name, s in [("адам жазған", human_text),
                ("кәдімгі модель", normal_text),
                ("су таңбалы модель", wm_text)]:
    z, share = z_score(s)
    print(f"{name:18} жасыл үлес: {share:.2f}   z = {z:5.2f}")
print("\nКӘДІМГІ:  ", prompt + normal_text[:130], "…")
print("СУ ТАҢБАЛЫ:", prompt + wm_text[:130], "…")

In [ ]:
length_rows = []
for n in [25, 50, 100, 200, 300]:
    length_rows.append((n, z_score(wm_text[:n])[0],
                        z_score(normal_text[:n])[0]))
    print(f"{n:3} таңба: су таңбалы z = {length_rows[-1][1]:5.2f}, "
          f"кәдімгі z = {length_rows[-1][2]:5.2f}")

rng = random.Random(0)
edit_rows = []
for frac in [0.0, 0.1, 0.2, 0.3, 0.5]:
    s = list(wm_text)
    for i in rng.sample(range(len(s)), int(frac * len(s))):
        s[i] = rng.choice(chars)                     # кездейсоқ таңба
    edit_rows.append((frac, z_score("".join(s))[0]))
    print(f"таңбалардың {frac:.0%} өзгертілді: z = {edit_rows[-1][1]:5.2f}")

In [ ]:
@torch.no_grad()
def quality_loss(s):
    ids = torch.tensor([encode(prompt + s)])
    logits = torch.cat([model(ids[:, max(0, i - 64):i])[0][:, -1]
                        for i in range(len(prompt), ids.shape[1])])
    return torch.nn.functional.cross_entropy(
        logits, ids[0, len(prompt):]).item()

delta_rows = []
for delta in [0.0, 0.5, 1.0, 2.0, 4.0]:
    texts = [generate(prompt, delta=delta, seed=s) for s in range(3)]
    z = sum(z_score(t)[0] for t in texts) / 3
    loss = sum(quality_loss(t) for t in texts) / 3
    delta_rows.append((delta, z, loss))
    print(f"δ = {delta}:  z = {z:5.2f},  шығын = {loss:.3f}")

## 20.4. Авторлық құқық

## 20.5. Заңдар: Қазақстан және әлем

## 20.6. ЖИ және энергия

## 20.7. Мектепте ЖИ-ді адал қолдану

## 20.8. Болашаққа көзқарас